# SESSION 17: SQL Project – HR Analysis (Restaurant & Reviews Analytics)

**Course:** Working with Databases & SQL Basics  
**Topic:** Relational Joins, Multi-Table Aggregation, and Partitioned Window Ranking  
**Database:** `restaurant_analytics.db`  
**Execution Environment:** Python 3.12, SQLite 3.49, Pandas 3.1

---

## 📌 Project Overview & Objectives
This capstone project tests comprehensive mastery of relational SQL design and analytics workflows. While listed in the coursework portal curriculum as "HR Analysis", the tasks focus on a production **Restaurant & Customer Reviews Analytics** data model (resembling platforms like Zomato, Swiggy, and Yelp).

### Key Technical Pillars:
1. **Relational Schema Definition**: Building normalized entities `Restaurant` and `Review` with primary and foreign key constraints.
2. **Categorical Aggregation**: Grouping and sorting restaurant distribution across cuisine categories (`GROUP BY`, `ORDER BY DESC`).
3. **Multi-Table Relational Joins**: Executing `INNER JOIN` across primary-foreign key relationships combined with multi-column `GROUP BY` and `AVG()` aggregations.
4. **Partitioned Window Ranking**: Deploying window ranking functions (`DENSE_RANK()` or `RANK()`) partitioned by cuisine to isolate top-rated regional competitors.

---

## Task 1: Creating `Restaurant` Table & Inserting Sample Records
> **Task 1:** Create a SQL table called `Restaurant` with columns: `id`, `name`, `cuisine`, `location`, and `average_rating`. Insert at least 5 sample rows representing popular restaurants from Zomato.

We define the `Restaurant` entity with constraints:
- `id`: `INTEGER PRIMARY KEY`
- `name`: `TEXT NOT NULL`
- `cuisine`: `TEXT NOT NULL`
- `location`: `TEXT NOT NULL`
- `average_rating`: `DECIMAL(3, 2) NOT NULL`

In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite database
db_path = "restaurant_analytics.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create Restaurant table
cursor.execute("DROP TABLE IF EXISTS Review;")
cursor.execute("DROP TABLE IF EXISTS Restaurant;")
cursor.execute("""
CREATE TABLE Restaurant (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    cuisine TEXT NOT NULL,
    location TEXT NOT NULL,
    average_rating DECIMAL(3, 2) NOT NULL
);
""")
conn.commit()

# Sample popular restaurants across multiple cuisine types
restaurants_data = [
    (1, "Punjab Grill", "North Indian", "Connaught Place", 4.50),
    (2, "Dhaba Estd 1986", "North Indian", "Cyber Hub", 4.30),
    (3, "Gulati", "North Indian", "Pandara Road", 4.70),
    (4, "Little Italy", "Italian", "Indiranagar", 4.40),
    (5, "Toscano", "Italian", "UB City", 4.60),
    (6, "Behrouz Biryani", "Mughlai", "Koramangala", 4.20),
    (7, "Biryani By Kilo", "Mughlai", "HSR Layout", 4.50)
]

cursor.executemany(
    "INSERT INTO Restaurant (id, name, cuisine, location, average_rating) VALUES (?, ?, ?, ?, ?);",
    restaurants_data
)
conn.commit()

# Query and display Restaurant table
df_restaurants = pd.read_sql_query("SELECT * FROM Restaurant ORDER BY id ASC;", conn)
print(f"[OK] Ingested {len(df_restaurants)} restaurants into 'Restaurant' table:")
display(df_restaurants)


[OK] Ingested 7 restaurants into 'Restaurant' table:


,id,name,cuisine,location,average_rating
0,1,Punjab Grill,North Indian,Connaught Place,4.5
1,2,Dhaba Estd 1986,North Indian,Cyber Hub,4.3
2,3,Gulati,North Indian,Pandara Road,4.7
3,4,Little Italy,Italian,Indiranagar,4.4
4,5,Toscano,Italian,UB City,4.6
5,6,Behrouz Biryani,Mughlai,Koramangala,4.2
6,7,Biryani By Kilo,Mughlai,HSR Layout,4.5


## Task 2: Cuisine Distribution Report
> **Task 2:** Write a SQL query to generate a report showing the number of restaurants for each cuisine type from your `Restaurant` table, ordered by the count in descending order.  
> *Hint: Use `GROUP BY` and `ORDER BY`.*

### Analytical Logic:
- `GROUP BY cuisine` buckets individual restaurants into their respective culinary segments.
- `COUNT(id)` tallies restaurant volume per cuisine.
- `ORDER BY restaurant_count DESC` highlights dominant platform categories.

In [2]:
query_task2 = """
SELECT 
    cuisine,
    COUNT(id) AS restaurant_count
FROM Restaurant
GROUP BY cuisine
ORDER BY restaurant_count DESC, cuisine ASC;
"""

df_cuisine_counts = pd.read_sql_query(query_task2, conn)
print("--- Cuisine Distribution Report ---")
display(df_cuisine_counts)


--- Cuisine Distribution Report ---


,cuisine,restaurant_count
0,North Indian,3
1,Italian,2
2,Mughlai,2


## Task 3: Creating `Review` Table & Ingesting Reviews
> **Task 3:** Add a new table called `Review` with columns: `id`, `restaurant_id`, `user_name`, `rating`, and `review_date`. Insert at least 10 sample reviews, linking them to restaurants using `restaurant_id`.

Schema definition:
- `id`: `INTEGER PRIMARY KEY`
- `restaurant_id`: `INTEGER NOT NULL REFERENCES Restaurant(id)`
- `user_name`: `TEXT NOT NULL`
- `rating`: `DECIMAL(3, 2) NOT NULL`
- `review_date`: `DATE NOT NULL`

In [3]:
# 1. Create Review table with foreign key linkage
cursor.execute("""
CREATE TABLE Review (
    id INTEGER PRIMARY KEY,
    restaurant_id INTEGER NOT NULL,
    user_name TEXT NOT NULL,
    rating DECIMAL(3, 2) NOT NULL,
    review_date DATE NOT NULL,
    FOREIGN KEY (restaurant_id) REFERENCES Restaurant(id)
);
""")
conn.commit()

# 2. Insert 14 realistic user reviews linking across all 7 restaurants
reviews_data = [
    (1,  3, "Aarav Sharma",   5.00, "2026-03-01"),
    (2,  3, "Diya Kapoor",    4.80, "2026-03-05"),
    (3,  5, "Rohan Verma",    4.80, "2026-03-02"),
    (4,  5, "Meera Nair",     4.60, "2026-03-06"),
    (5,  1, "Kabir Mehta",    4.50, "2026-03-03"),
    (6,  1, "Ananya Sen",     4.70, "2026-03-07"),
    (7,  7, "Siddharth Rao",  4.60, "2026-03-04"),
    (8,  7, "Pooja Hegde",    4.40, "2026-03-08"),
    (9,  4, "Vikram Singhal", 4.40, "2026-03-02"),
    (10, 4, "Ishaan Goel",    4.20, "2026-03-09"),
    (11, 2, "Neha Gupta",     4.20, "2026-03-01"),
    (12, 2, "Kunal Roy",      4.40, "2026-03-10"),
    (13, 6, "Tanvi Joshi",    4.10, "2026-03-04"),
    (14, 6, "Aditya Malik",   4.30, "2026-03-11")
]

cursor.executemany(
    "INSERT INTO Review (id, restaurant_id, user_name, rating, review_date) VALUES (?, ?, ?, ?, ?);",
    reviews_data
)
conn.commit()

# Query and display Review table preview
df_reviews = pd.read_sql_query("SELECT * FROM Review ORDER BY id ASC;", conn)
print(f"[OK] Ingested {len(df_reviews)} reviews into 'Review' table:")
display(df_reviews)


[OK] Ingested 14 reviews into 'Review' table:


,id,restaurant_id,user_name,rating,review_date
0,1,3,Aarav Sharma,5.0,2026-03-01
1,2,3,Diya Kapoor,4.8,2026-03-05
2,3,5,Rohan Verma,4.8,2026-03-02
3,4,5,Meera Nair,4.6,2026-03-06
4,5,1,Kabir Mehta,4.5,2026-03-03
5,6,1,Ananya Sen,4.7,2026-03-07
6,7,7,Siddharth Rao,4.6,2026-03-04
7,8,7,Pooja Hegde,4.4,2026-03-08
8,9,4,Vikram Singhal,4.4,2026-03-02
9,10,4,Ishaan Goel,4.2,2026-03-09


## Task 4: Average Review Rating per Restaurant (JOIN & GROUP BY)
> **Task 4:** Write a SQL query using a `JOIN` to display each restaurant's `name`, `cuisine`, and its average review rating (from the `Review` table), ordered by highest average rating first.  
> *Hint: Use `JOIN` and `GROUP BY` with aggregate functions.*

### Relational Join Mechanics:
1. `Restaurant r INNER JOIN Review rev ON r.id = rev.restaurant_id` connects each review to its parent restaurant profile.
2. `GROUP BY r.id, r.name, r.cuisine` segments the joined dataset by unique restaurant entity.
3. `AVG(rev.rating)` dynamically aggregates real customer scores.
4. `ORDER BY avg_review_rating DESC` ranks top-rated dining destinations first.

In [4]:
query_task4 = """
SELECT 
    r.name,
    r.cuisine,
    ROUND(AVG(rev.rating), 2) AS avg_review_rating
FROM Restaurant r
INNER JOIN Review rev ON r.id = rev.restaurant_id
GROUP BY r.id, r.name, r.cuisine
ORDER BY avg_review_rating DESC, r.name ASC;
"""

df_avg_ratings = pd.read_sql_query(query_task4, conn)
print("--- Average Review Rating per Restaurant ---")
display(df_avg_ratings)


--- Average Review Rating per Restaurant ---


,name,cuisine,avg_review_rating
0,Gulati,North Indian,4.9
1,Toscano,Italian,4.7
2,Punjab Grill,North Indian,4.6
3,Biryani By Kilo,Mughlai,4.5
4,Dhaba Estd 1986,North Indian,4.3
5,Little Italy,Italian,4.3
6,Behrouz Biryani,Mughlai,4.2


## Task 5: Ranking Restaurants within Each Cuisine using Window Functions
> **Task 5:** Use a window function to rank restaurants by their average review rating within each cuisine type, showing the restaurant name, cuisine, average rating, and rank.  
> *Hint: Use the `RANK()` or `DENSE_RANK()` window function partitioned by cuisine.*

### Analytical & Window Specification:
- We compute restaurant averages using a Common Table Expression (`WITH RestaurantAverages AS (...)`).
- Window Function: `DENSE_RANK() OVER (PARTITION BY cuisine ORDER BY average_rating DESC)`.
- `PARTITION BY cuisine`: Resets the ranking counter whenever the cuisine changes, allowing fair intra-category benchmarking.
- `ORDER BY average_rating DESC`: Assigns Rank 1 to the highest-rated restaurant within that specific cuisine.

In [5]:
query_task5 = """
WITH RestaurantAverages AS (
    SELECT 
        r.name,
        r.cuisine,
        ROUND(AVG(rev.rating), 2) AS average_rating
    FROM Restaurant r
    INNER JOIN Review rev ON r.id = rev.restaurant_id
    GROUP BY r.id, r.name, r.cuisine
)
SELECT 
    name,
    cuisine,
    average_rating,
    DENSE_RANK() OVER (
        PARTITION BY cuisine 
        ORDER BY average_rating DESC
    ) AS rank
FROM RestaurantAverages
ORDER BY cuisine ASC, rank ASC;
"""

df_ranked = pd.read_sql_query(query_task5, conn)
print("--- Cuisine-Partitioned Restaurant Leaderboard ---")
display(df_ranked)

# Close database connection
conn.close()
print("\n[OK] Database connection closed cleanly.")


--- Cuisine-Partitioned Restaurant Leaderboard ---


,name,cuisine,average_rating,rank
0,Toscano,Italian,4.7,1
1,Little Italy,Italian,4.3,2
2,Biryani By Kilo,Mughlai,4.5,1
3,Behrouz Biryani,Mughlai,4.2,2
4,Gulati,North Indian,4.9,1
5,Punjab Grill,North Indian,4.6,2
6,Dhaba Estd 1986,North Indian,4.3,3



[OK] Database connection closed cleanly.


---

## 🎯 Summary of Key Findings & Relational Insights

| Cuisine | Rank 1 Leader | Average Review Score | Operational Insight |
| :--- | :--- | :---: | :--- |
| **Italian** | **Toscano** | **4.70** | Leading Italian brand on platform; outperforms Little Italy (4.30). |
| **Mughlai** | **Biryani By Kilo** | **4.50** | Strong customer loyalty; edges out Behrouz Biryani (4.20). |
| **North Indian** | **Gulati** | **4.90** | Platform top-rated restaurant overall (near-perfect 4.90 rating). |
